In [1]:
# pmm_affine_qd.py
# Minimal Affine Eigenvalue PMM (AE‑PMM) for mapping ω → (E0, E1, ...)
# Optional observable head for ⟨r^2⟩ with PSD constraint.
#
# Usage (quick):
#   1) Prepare a training set dict with keys:
#        data = {
#           'omega': torch.tensor([...], dtype=torch.float64),            # shape (B,)
#           'E_targets': torch.tensor([[E0,E1], ...], dtype=torch.float64) # shape (B, K)
#           # optional: 'r2_targets': torch.tensor([[r2_gs, r2_es], ...], dtype=torch.float64)
#        }
#   2) Run train() at the bottom (uncomment example). It will save 'pmm_qd.pth'.
#   3) Use predict() to get energies (and ⟨r^2⟩) for new ω values.
#
# Notes:
#   • The operator is M(ω) = Σ_j φ_j(ω) A_j, where φ_j are polynomial features [1, ω, ω^2, ...].
#   • A_j are *learned* symmetric matrices (Hermitian in ℝ).
#   • E(ω) are the lowest K eigenvalues of M(ω), from torch.linalg.eigh (batched).
#   • Optional observable matrices O_m are parameterized PSD: O_m = C_m C_m^T (guarantees non‑negative expectations).
#   • Smooth eigenvector tracking penalty encourages consistent labeling across neighboring ω points.

from __future__ import annotations
import math
from dataclasses import dataclass
from typing import Optional, Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

try:
    from tqdm import trange
except Exception:
    # Fallback if tqdm is unavailable
    def trange(n, **kwargs):
        return range(n)

# -----------------------
# Utilities
# -----------------------

def set_seed(seed: int = 0):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def feature_map_omega(omega: Tensor, degree: int = 2) -> Tensor:
    """
    Polynomial features φ(ω) = [1, ω, ω^2, ..., ω^degree].
    Args:
        omega: shape (B,) or (B,1)
        degree: non-negative int
    Returns:
        φ: shape (B, degree+1)
    """
    omega = omega.view(-1, 1)
    feats = [torch.ones_like(omega)]
    for d in range(1, degree + 1):
        feats.append(omega ** d)
    return torch.cat(feats, dim=1)  # (B, m)


# -----------------------
# Affine PMM core
# -----------------------

class AffinePMM(nn.Module):
    """
    Affine Parametric Matrix Model (real-symmetric version) for ω-only.
    M(ω) = Σ_j φ_j(ω) * A_j, with A_j learned and symmetrized.
    Optional observable heads: O_m = C_m C_m^T (PSD) for expectations.
    """

    def __init__(
        self,
        n_latent: int = 10,       # latent operator dimension (n × n)
        degree: int = 2,          # polynomial degree for ω features
        n_obs: int = 0,           # number of observable heads (e.g., 1 for ⟨r^2⟩)
        dtype: torch.dtype = torch.float64,
        device: Optional[torch.device] = None,
    ):
        super().__init__()
        self.n = n_latent
        self.degree = degree
        self.m = degree + 1           # number of φ_j basis functions
        self.n_obs = n_obs

        # A_params: (m, n, n) unconstrained → symmetric via (A + A^T)/2
        A_init = torch.randn(self.m, self.n, self.n, dtype=dtype, device=device) * (1.0 / math.sqrt(self.n))
        self.A_params = nn.Parameter(A_init)

        # Observable heads: C_params (n_obs, n, n), O = C C^T is PSD
        if n_obs > 0:
            C_init = torch.randn(self.n_obs, self.n, self.n, dtype=dtype, device=device) * (1.0 / math.sqrt(self.n))
            self.C_params = nn.Parameter(C_init)
        else:
            self.register_parameter('C_params', None)

    def symmetricize(self, A: Tensor) -> Tensor:
        return 0.5 * (A + A.transpose(-1, -2))

    def build_M(self, omega: Tensor) -> Tensor:
        """Build batched operator M(ω) with shape (B, n, n)."""
        phi = feature_map_omega(omega, degree=self.degree).to(self.A_params.dtype)
        # Symmetrize A_j once (m, n, n)
        A = self.symmetricize(self.A_params)
        # Weighted sum over m: M_b = Σ_j φ_bj A_j
        # (B, m) @ (m, n, n) → (B, n, n)
        M = torch.einsum('bm,mij->bij', phi, A)
        return M

    def build_observables(self) -> Optional[Tensor]:
        if self.n_obs == 0 or self.C_params is None:
            return None
        # O_m = C_m C_m^T (n_obs, n, n)
        C = self.C_params
        O = torch.einsum('mij,mkj->mik', C, C)  # (n_obs, n, n)
        return O

    @torch.no_grad()
    def spectrum(self, omega: Tensor, k: int = 2) -> Tuple[Tensor, Tensor]:
        """Return lowest-k eigenpairs. evals: (B, k), evecs: (B, n, k)."""
        M = self.build_M(omega)  # (B,n,n)
        evals, evecs = torch.linalg.eigh(M)  # ascending
        return evals[:, :k], evecs[:, :, :k]

    def forward(self, omega: Tensor, k: int = 2) -> Dict[str, Tensor]:
        M = self.build_M(omega)  # (B,n,n)
        evals, evecs = torch.linalg.eigh(M)  # (B,n), (B,n,n)
        evals_k = evals[:, :k]
        evecs_k = evecs[:, :, :k]

        out = {'E': evals_k, 'M': M, 'V': evecs_k}
        O = self.build_observables()
        if O is not None:
            # Expectation per obs & state: ⟨v|O|v⟩, shapes (B, n_obs, k)
            # Expand to (B, n, k) for V, (n_obs, n, n) for O
            v = evecs_k  # (B,n,k)
            vT = v.transpose(1, 2)  # (B,k,n)
            # Compute vT O v for each obs: (B, n_obs, k)
            # First O v → (n_obs, B, n, k)
            Ov = torch.einsum('mij,bjk->mbik', O, v)
            vT_O_v = torch.einsum('bki,mbik->bmk', vT, Ov)
            out['obs'] = vT_O_v  # (B, n_obs, k)
        return out


# -----------------------
# Training
# -----------------------

@dataclass
class TrainCfg:
    n_latent: int = 10
    degree: int = 2          # φ = [1, ω, ω^2]
    k_states: int = 2
    n_obs: int = 0           # set to 1 if supervising ⟨r^2⟩
    lr: float = 1e-2
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    epochs: int = 4000
    seed: int = 0
    device: str = 'cpu'
    dtype: torch.dtype = torch.float64

    # Loss weights
    w_energy: float = 1.0
    w_obs: float = 0.2
    w_overlap: float = 0.01  # smooth eigenvector tracking
    # Regularize operator size
    w_frob: float = 1e-6


def overlap_penalty(evecs: Tensor) -> Tensor:
    """
    Encourage smooth eigenvector labeling along sorted ω.
    evecs: (B, n, k) matching sorted ω order in the *batch*.
    Returns: scalar penalty ~ mean over t of 1 - |⟨v_t, v_{t+1}⟩| for all states.
    """
    B, n, k = evecs.shape
    if B <= 1:
        return evecs.new_tensor(0.0)
    v_t = evecs[:-1]      # (B-1,n,k)
    v_tp1 = evecs[1:]     # (B-1,n,k)
    # Overlap for each state j: |v_t[:, :, j]^T v_{t+1}[:, :, j]|
    num = torch.einsum('bni,bni->bn', v_t, v_tp1)  # (B-1, k) after selecting each j separately via batch matmul trick
    # The einsum above sums over n for each k implicitly because we didn't separate k; fix by reshaping
    # Let's compute per-state explicitly for clarity
    v_t_rs = v_t.transpose(1, 2)     # (B-1,k,n)
    v_tp1_rs = v_tp1.transpose(1, 2) # (B-1,k,n)
    ov = torch.sum(v_t_rs * v_tp1_rs, dim=-1).abs()  # (B-1,k)
    pen = (1.0 - ov).mean()
    return pen


def train(model: AffinePMM, data: Dict[str, Tensor], cfg: TrainCfg) -> Dict[str, Tensor]:
    device = torch.device(cfg.device)
    model.to(device=device, dtype=cfg.dtype)

    omega = data['omega'].to(device=device, dtype=cfg.dtype).view(-1)
    E_tgt = data['E_targets'].to(device=device, dtype=cfg.dtype)  # (B,K)
    assert E_tgt.ndim == 2 and E_tgt.size(1) >= cfg.k_states, "E_targets must have at least k_states columns"

    has_obs = ('r2_targets' in data) and (cfg.n_obs > 0)
    if has_obs:
        r2_tgt = data['r2_targets'].to(device=device, dtype=cfg.dtype)  # (B,K) or (B,)
        if r2_tgt.ndim == 1:
            r2_tgt = r2_tgt[:, None]  # (B,1)

    # Sort by ω for stable overlap penalty
    sort_idx = torch.argsort(omega)
    omega = omega[sort_idx]
    E_tgt = E_tgt[sort_idx]
    if has_obs:
        r2_tgt = r2_tgt[sort_idx]

    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    for _ in trange(cfg.epochs, desc='Train PMM'):
        opt.zero_grad(set_to_none=True)
        out = model(omega, k=cfg.k_states)
        E_pred = out['E']           # (B,K)
        V = out['V']                # (B,n,K)

        loss_e = F.mse_loss(E_pred, E_tgt[:, :cfg.k_states])
        loss = cfg.w_energy * loss_e

        if has_obs and 'obs' in out:
            obs_pred = out['obs'][:, :cfg.n_obs, :cfg.k_states]  # (B, n_obs, K)
            # if r2_tgt provided as (B,K) and n_obs==1, align shapes:
            if r2_tgt.ndim == 2 and cfg.n_obs == 1:
                loss_o = F.mse_loss(obs_pred[:, 0, :], r2_tgt[:, :cfg.k_states])
            else:
                # generic broadcasted MSE
                loss_o = F.mse_loss(obs_pred, r2_tgt)
            loss = loss + cfg.w_obs * loss_o
        else:
            loss_o = torch.zeros((), dtype=cfg.dtype, device=device)

        # Smooth eigenvector tracking across neighboring ω
        loss_overlap = overlap_penalty(V)
        loss = loss + cfg.w_overlap * loss_overlap

        # Frobenius norm regularizer on A_params to keep spectra tame
        frob = torch.sum(model.A_params ** 2)
        loss = loss + cfg.w_frob * frob

        loss.backward()
        if cfg.grad_clip is not None and cfg.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

    return {'omega_sorted': omega.detach().cpu(),
            'E_pred': E_pred.detach().cpu(),
            'E_tgt_sorted': E_tgt[:, :cfg.k_states].detach().cpu()}


@torch.no_grad()
def predict(model: AffinePMM, omega_new: Tensor, k: int = 2, device: str = 'cpu') -> Dict[str, Tensor]:
    model.eval()
    omega_new = omega_new.to(device=model.A_params.device, dtype=model.A_params.dtype)
    E, V = model.spectrum(omega_new, k=k)
    out = {'omega': omega_new.detach().cpu(), 'E': E.detach().cpu()}
    O = model.build_observables()
    if O is not None:
        # replicate forward observable computation
        v = V  # (B,n,k)
        Ov = torch.einsum('mij,bjk->mbik', O, v)
        vT_O_v = torch.einsum('bki,mbik->bmk', v.transpose(1, 2), Ov)  # (B,n_obs,k)
        out['obs'] = vT_O_v.detach().cpu()
    return out

In [2]:
# Minimal AE‑PMM demo for ω=1.0 with target ground‑state energy E0=3.0
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# ---- Features ----
def feature_map_omega(omega: torch.Tensor, degree: int = 2) -> torch.Tensor:
    omega = omega.view(-1, 1)
    feats = [torch.ones_like(omega)]
    for d in range(1, degree + 1):
        feats.append(omega ** d)
    return torch.cat(feats, dim=1)


# ---- AE‑PMM core (real‑symmetric) ----
class AffinePMM(nn.Module):
    def __init__(self, n_latent: int = 6, degree: int = 2, dtype=torch.float64, device="cpu"):
        super().__init__()
        self.n = n_latent
        self.degree = degree
        self.m = degree + 1
        A_init = torch.randn(self.m, self.n, self.n, dtype=dtype, device=device) / math.sqrt(self.n)
        self.A_params = nn.Parameter(A_init)

    def symmetricize(self, A: torch.Tensor) -> torch.Tensor:
        return 0.5 * (A + A.transpose(-1, -2))

    def build_M(self, omega: torch.Tensor) -> torch.Tensor:
        phi = feature_map_omega(omega, self.degree).to(self.A_params.dtype)
        A = self.symmetricize(self.A_params)                           # (m,n,n)
        M = torch.einsum("bm,mij->bij", phi, A)                        # (B,n,n)
        return M

    def forward(self, omega: torch.Tensor, k: int = 1):
        M = self.build_M(omega)
        evals, evecs = torch.linalg.eigh(M)                            # ascending
        return {"E": evals[:, :k], "M": M, "V": evecs[:, :, :k]}


# ---- Train on a single (ω, E0) pair ----
omega = torch.tensor([1.0], dtype=torch.float64)
E_targets = torch.tensor([[3.0]], dtype=torch.float64)  # shape (B=1, K=1)

model = AffinePMM(n_latent=6, degree=2, dtype=torch.float64, device="cpu")
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

def step():
    opt.zero_grad(set_to_none=True)
    out = model(omega, k=1)
    loss = F.mse_loss(out["E"], E_targets)
    loss.backward()
    opt.step()
    return float(loss.detach())

# Train briefly (single point fits quickly)
loss_hist = []
for it in range(1500):
    loss_hist.append(step())

with torch.no_grad():
    pred = model(omega, k=1)["E"].squeeze().item()

print(f"Target E0(ω=1.0) = 3.0")
print(f"Predicted E0(ω=1.0) after training = {pred:.6f}")
print(f"Final MSE = {loss_hist[-1]:.6e}")
print("A quick sanity check: training on a single point is underdetermined,")
print("so many operators M(ω) can fit it perfectly; this demo just shows the pipeline works.")


Target E0(ω=1.0) = 3.0
Predicted E0(ω=1.0) after training = 2.999700
Final MSE = 8.324393e-08
A quick sanity check: training on a single point is underdetermined,
so many operators M(ω) can fit it perfectly; this demo just shows the pipeline works.
